# Bring data from P1 and add necessary features

This code will take the outputs from the first paper in this thesis, add the necessary columns, and save the files in the correct folder.

## 1. Import packages
First, packages will be imported and configured.

In [ ]:
!pip install pandas numpy openpyxl

In [ ]:
# Import packages
import pandas as pd
import os
import numpy as np

## 2. Get list with files to read and other necessary data

In [ ]:
## Get list with files to read

# State path where processed reviews are
p1_path = "./01_outputs_p1/01_Reviews"
data_path = "./02_other_inputs_preprocessing/01_State and company data/"
out_path = "./01_outputs_p1/01_Reviews/"

# Get list with files to read
files_reviews = [file for file in os.listdir(p1_path) if "_reviews.csv" in file]

In [ ]:
## Other data

# Read state and company data
state_df = pd.read_excel(data_path + "State and company information.xlsx", "Consolidated state data")
companies_df = pd.read_excel(data_path + "State and company information.xlsx", "Consolidated company data")
stock_df = pd.read_excel(data_path + "State and company information.xlsx", "Consolid. stock hist mean")

# Read state mapping
state_mapping_df = pd.read_csv(data_path + "state_mapping.csv")
state_mapping_dict = state_mapping_df.set_index('STATE')['STATE SYMBOL'].to_dict()

## 3. Add data and save

For each company, the following steps are followed:
 - Get corrected state. First, the reviews with missing values in the author_location column will use the company HQ's state. For locations which are not formated correctly, the most common state will be used.
 - State, company and stock information will be added.

In [ ]:
# Define function to get state
def get_state(location):
    try:
        return location.split(",")[1].strip().upper()
    except:
        return location
    
# Define function to transform state to correct format
def get_state_format(state, state_dict):
    if state in state_dict.keys():
        return state_dict[state]
    
    else:
        return state

In [ ]:
# Iterate through all files
for file in files_reviews:
    
    ## Start loop checks

    # Get company ticker
    company_ticker = file.split("_")[0]

    # Check if company has already been done
    if company_ticker + "_reviews.csv" in [file for file in os.listdir(out_path) if "_reviews.csv" in file]:

        #print(file + " already done")
        continue

    try:

        ## Read files

        # Read file
        print(file + " starting")
        df = pd.read_csv(p1_path + "/" + file)

        # Add company column
        df['Company'] = company_ticker


        ## Correct author location
        
        # Add author location filled with HQ
        company_hq = companies_df[companies_df['Ticker']==company_ticker].reset_index().at[0, 'Headquarters Location']
        df['author_location_filledhq'] = df['author_location'].apply(lambda al: al if al != "Missing value" else company_hq)

        # Get state
        df['author_location_filledhq_state'] = df['author_location_filledhq'].apply(get_state)
        df['author_location_filledhq_state'] = df['author_location_filledhq_state'].apply(lambda st: get_state_format(st, state_mapping_dict))
        
        # Set most popular value
        popular_state = df['author_location_filledhq_state'].value_counts().idxmax()
        df['author_location_filledhq_state'] = np.where(df['author_location_filledhq_state'].isin(state_mapping_dict.values()), df['author_location_filledhq_state'], popular_state)


        ## Add state, company, stock and sp500 information

        # Add state information
        df = pd.merge(df, state_df, left_on=['author_location_filledhq_state'], right_on=['Acronym'], how='left')

        # Add company information
        df = pd.merge(df, companies_df, left_on=['Company'], right_on=['Ticker'], how='left')
        
        # Add stock information
        df['month'] = pd.to_datetime(df['date']).dt.month
        df['year'] = pd.to_datetime(df['date']).dt.year

        stock_df_melt = pd.melt(stock_df, ['Month', 'Year'])
        stock_df_melt_month = stock_df_melt[stock_df_melt['variable'] == 'Close_' + company_ticker + '_previousmonth'].rename(columns={'value': 'stock_percchange_month'})
        stock_df_melt_year = stock_df_melt[stock_df_melt['variable'] == 'Close_' + company_ticker + '_previousyear'].rename(columns={'value': 'stock_percchange_year'})

        df = pd.merge(df, stock_df_melt_month[['Month', 'Year', 'stock_percchange_month']], left_on=['month', 'year'], right_on=['Month', 'Year'], how='left').drop(['Month', 'Year'], axis=1)
        df = pd.merge(df, stock_df_melt_year[['Month', 'Year', 'stock_percchange_year']], left_on=['month', 'year'], right_on=['Month', 'Year'], how='left').drop(['Month', 'Year'], axis=1)

        # Add sp500 information
        stock_df_melt_month_sp500 = stock_df_melt[stock_df_melt['variable'] == 'Close_sp500_previousmonth'].rename(columns={'value': 'sp500_percchange_month'})
        stock_df_melt_year_sp500 = stock_df_melt[stock_df_melt['variable'] == 'Close_sp500_previousyear'].rename(columns={'value': 'sp500_percchange_year'})

        df = pd.merge(df, stock_df_melt_month_sp500[['Month', 'Year', 'sp500_percchange_month']], left_on=['month', 'year'], right_on=['Month', 'Year'], how='left').drop(['Month', 'Year'], axis=1)
        df = pd.merge(df, stock_df_melt_year_sp500[['Month', 'Year', 'sp500_percchange_year']], left_on=['month', 'year'], right_on=['Month', 'Year'], how='left').drop(['Month', 'Year'], axis=1)


        ## Save and report success
        df.to_csv(out_path + file)
        print("\t"+ file + " successful")

    except:
        print(file + " NOT SUCCESSFUL")